In [2]:
import pandahouse
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
%matplotlib inline

In [3]:
connection = {
    'host' : 'http://clickhouse.lab.karpov.courses:8123',
    'password' : 'dpo_python_2020',
    'user' : 'student',
    'database' : 'simulator_20260320'
}

## Описание задания
    Относительно недавно (в 2018-м году) исследователи из Яндекса разработали метод анализа тестов над метриками-отношениями (такие как CTR). Идея метода заключается в следующем:

Вместо того, чтобы анализировать в тестах «поюзерные» CTR, можно использовать другую метрику, и при этом гарантируется (в отличие от сглаженного CTR), что если тест на этой другой метрике «прокрасится» и увидит изменения, значит изменения есть и в метрике исходной (то есть в лайках на пользователя и в пользовательских CTR). Фактически получившаяся метрика гарантированно будет меняться в том же направлении, что и "общий" CTR по группам. То есть мы можем делать выводы по "общему" CTR, не прибегая к таким тяжёлым вычислительным методам как бутстрап. Такой метрикой является метрика 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠: <br>

1) Считаем общий CTR в контрольной группе 𝐶𝑇𝑅𝑐𝑜𝑛𝑡𝑟𝑜𝑙=𝑠𝑢𝑚(𝑙𝑖𝑘𝑒𝑠)/𝑠𝑢𝑚(𝑣𝑖𝑒𝑤𝑠) 
2) Посчитаем в обеих группах поюзерную метрику 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠=𝑙𝑖𝑘𝑒𝑠−𝐶𝑇𝑅𝑐𝑜𝑛𝑡𝑟𝑜𝑙∗𝑣𝑖𝑒𝑤𝑠 
3) После чего сравним t-тестом отличия в группах по метрике 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠  
При этом гарантируется, что при приличном размере выборки можно увеличить чувствительность вашей метрики (или, по крайней мере, не сделать хуже).

Задачи, которые будем решать на следующих шагах, такие:

1) Проанализируйте тест между группами 0 (контроль) и 3(экспериментальная, новый рекомендательный алгоритм) по метрике линеаризованных лайков. Видно ли отличие? Стало ли 𝑝−𝑣𝑎𝑙𝑢𝑒 меньше по сравнению с обычным CTR?
2) Проанализируйте тест между группами 1(контроль) и 2(экспериментальная, другой новый рекомендательный алгоритм) по метрике линеаризованных лайков. Видно ли отличие? Стало ли 𝑝−𝑣𝑎𝑙𝑢𝑒 меньше по сравнению с обычным CTR?
Данные берём в том же диапазоне, в котором проводился A/B-тест: с 2026-02-27 по 2026-03-05 включительно.

### Задание 1
Пусть группы 0 и 1 — это наши контрольные группы. Чему равен в них общегрупповой CTR (в виде доли)? Укажите значение до 2 знака после точки.

In [4]:
q = """
SELECT exp_group,
    user_id,
    sum(action = 'like') as likes,
    sum(action = 'view') as views,
    likes/views as ctr
FROM {db}.feed_actions
WHERE toDate(time) between '2026-02-27' and '2026-03-05'
    and exp_group in (0,1)
GROUP BY exp_group, user_id
"""
df = pandahouse.read_clickhouse(q, connection=connection)

In [11]:
ctr_1 = df[df.exp_group==1].likes.sum()/df[df.exp_group==1].views.sum()
ctr_0 =df[df.exp_group==0].likes.sum()/df[df.exp_group==0].views.sum()
print(f'CTR group 1={ctr_1:.2f}\nCTR group 0={ctr_0:.2f}')

CTR group 1=0.21
CTR group 0=0.21


### Задание 2
Начнём с групп 0 и 3! Давайте сравним их t-тестом — сначала возьмём обычный пользовательский CTR, а затем линеаризованные лайки. 

In [12]:
q = """
SELECT exp_group,
    user_id,
    sum(action = 'like') as likes,
    sum(action = 'view') as views,
    likes/views as ctr
FROM {db}.feed_actions
WHERE toDate(time) between '2026-02-27' and '2026-03-05'
    and exp_group in (0,3)
GROUP BY exp_group, user_id
"""
df = pandahouse.read_clickhouse(q, connection=connection)

In [13]:
#t-test на обычном поюзерном CTR
stats.ttest_ind(df[df.exp_group==0].ctr, 
                df[df.exp_group==3].ctr,
                equal_var=False)

Ttest_indResult(statistic=-13.935320516755773, pvalue=6.216047483062228e-44)

In [14]:
#t-test на линеаризованных лайках
#Считаем общий CTR в контрольной группе  
CTRcontrol= df[df.exp_group==0].likes.sum()/df[df.exp_group==0].views.sum()

#Посчитаем в обеих группах поюзерную метрику 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠=𝑙𝑖𝑘𝑒𝑠−𝐶𝑇𝑅𝑐𝑜𝑛𝑡𝑟𝑜𝑙∗𝑣𝑖𝑒𝑤𝑠
df['linearized_likes']= df.likes - CTRcontrol*df.views

#После чего сравним  t-тестом отличия в группах по метрике 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠
stats.ttest_ind(df[df.exp_group==0].linearized_likes, 
                df[df.exp_group==3].linearized_likes,
                equal_var=False)

Ttest_indResult(statistic=-16.186230032932844, pvalue=1.4918137745326139e-58)

**Вывод:**<br>
P-value для пользовательского CTR оказалось примерно равным 6.216047483062228e-44. <br>
P-value для линеаризованных лайков оказалось примерно равным 1.4918137745326139e-58. <br>
Таким образом, линеаризация уменьшила p-value. <br>
Различия между группами статистически значимы для пользовательского CTR, а для линеаризованной метрики они тоже значимы.<br>

### Задание 3

Сделаем то же самое для групп 1 и 2! Что у нас получилось?

In [15]:
q = """
SELECT exp_group,
    user_id,
    sum(action = 'like') as likes,
    sum(action = 'view') as views,
    likes/views as ctr
FROM {db}.feed_actions
WHERE toDate(time) between '2026-02-27' and '2026-03-05'
    and exp_group in (1,2)
GROUP BY exp_group, user_id
"""
df = pandahouse.read_clickhouse(q, connection=connection)

print('пользовательский CTR',stats.ttest_ind(df[df.exp_group==1].ctr, 
                df[df.exp_group==2].ctr,
                equal_var=False))

#Считаем общий CTR в контрольной группе  
CTRcontrol= df[df.exp_group==1].likes.sum()/df[df.exp_group==1].views.sum()

#Посчитаем в обеих группах поюзерную метрику 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠=𝑙𝑖𝑘𝑒𝑠−𝐶𝑇𝑅𝑐𝑜𝑛𝑡𝑟𝑜𝑙∗𝑣𝑖𝑒𝑤𝑠
df['linearized_likes']= df.likes - CTRcontrol*df.views

#После чего сравним  t-тестом отличия в группах по метрике 𝑙𝑖𝑛𝑒𝑎𝑟𝑖𝑧𝑒𝑑_𝑙𝑖𝑘𝑒𝑠
print('линеаризованные лайки',stats.ttest_ind(df[df.exp_group==1].linearized_likes, 
                df[df.exp_group==2].linearized_likes,
                equal_var=False))

пользовательский CTR Ttest_indResult(statistic=0.4051491913112757, pvalue=0.685373331140751)
линеаризованные лайки Ttest_indResult(statistic=5.936377101934482, pvalue=2.9805064038667734e-09)


**Вывод:** <br>
P-value для пользовательского CTR оказалось примерно равным 0.68. <br>
P-value для линеаризованных лайков оказалось примерно равным 2.9805064038668164e-09. <br>
Таким образом, линеаризация уменьшила p-value.<br>
Различия между группами статистически незначимы для пользовательского CTR, а для линеаризованной метрики они стали значимыми.

### Какой итоговый вывод мы можем сделать по поводу линеаризации?

Линеаризация — это очень быстрый метод, также линеаризация увеличила чувствительность нашего t-теста. При этом фактически проверяет ту же гипотезу, что и пуассоновский бутстрап, сравнивая глобальные CTR. Резюмируя, этот метод отлично подходит для анализа метрик-отношений. Полученная метрика является сонаправленной с глобальным CTR — то есть если глобальный CTR растёт, то и эта метрика растёт, и наоборот. Это делает метод гораздо более ресурсоэффективной заменой пуассоновскому бутстрапу. 